# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sameer-159/FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#### Task Type: Classification

##### Problem Statement:
Predict whether a piece of content is likely to decline in performance and therefore should be reviewed for refresh.

##### Why Classification?
The outcome is a category:

*  Needs Refresh (1)
* Does Not Need Refresh (0)

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The ideal target would be a historical label showing whether a content item was refreshed and improved afterward.

Since that label is not available, I use a proxy target.

Proxy Target:
- Content with a declining trend = Needs Refresh (1)
- Content with a stable or growing trend = Does Not Need Refresh (0)

This proxy assumes that declining content is more likely to require intervention.

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv('/content/content_refresh_anonymized.csv')

In [15]:
df["needs_refresh"] = (df["trend_direction"] == "declining").astype(int)

df[["trend_direction", "needs_refresh"]].head()

,trend_direction,needs_refresh
0,down,0
1,down,0
2,down,0
3,stable,0
4,down,0


In [7]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,needs_refresh
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0


 #### Proxy Explanation:

A declining trend suggests the content is losing performance and may benefit from intervention.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: F1 Score

I selected F1 Score because both precision and recall are important.

Why precision matters:
- We do not want to waste editor time reviewing content that is actually healthy.

Why recall matters:
- We do not want to miss content that genuinely needs a refresh.

A balanced F1 score ensures both concerns are considered.

Other possible metrics:
- Accuracy
- Precision
- Recall
- ROC-AUC

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content item.

Each row in the dataset represents a single piece of content and contains information about its performance.

Examples of features may include:
- Views
- Click-through rate
- Engagement metrics
- Trend indicators

The model will make one prediction per content item.

In [16]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 45


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,needs_refresh
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0


In [11]:
df.shape

(30000, 45)

Each row corresponds to a single piece of content and its performance characteristics.

One row = one content item.

This is the level at which predictions and decisions will be made.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule such as "refresh content when CTR is below a threshold" only considers one variable.

However, content performance depends on multiple factors including:

- CTR
- Average Position
- Engagement Rate
- Search Volume
- Competition
- Trend Direction
- AI Traffic Percentage

The examples above show cases where the rule makes incorrect decisions.

Machine learning can learn relationships between all these variables simultaneously and identify patterns that a single threshold cannot capture.

Therefore, ML is more suitable than a fixed rule for deciding which content should be refreshed.

In [20]:
median_ctr = df["ctr"].median()

df["rule_prediction"] = (
    df["ctr"] < median_ctr
).astype(int)

df[
    [
        "content_id",
        "ctr",
        "needs_refresh",
        "rule_prediction"
    ]
].head(10)

,content_id,ctr,needs_refresh,rule_prediction
0,content_304f48230142,0.76,0,0
1,content_a1fb4e703a9e,0.05,0,1
2,content_9aa793d4d895,0.09,0,0
3,content_331d6c4de07b,0.49,0,0
4,content_d99b7a2d90ca,0.13,0,0
5,content_d4084a4bc775,0.03,0,1
6,content_9a34b442b552,0.00,0,1
7,content_a63219c6e95a,0.06,0,1
8,content_5e6c160719bc,0.09,0,0
9,content_c27558df2b0c,0.16,0,0


In [21]:
rule_accuracy = (
    df["rule_prediction"] == df["needs_refresh"]
).mean()

print(f"Rule Accuracy: {rule_accuracy:.2%}")

Rule Accuracy: 50.63%


In [22]:
# Find examples where the rule fails

df[
    df["rule_prediction"] != df["needs_refresh"]
][
    [
        "content_id",
        "ctr",
        "engagement_rate",
        "avg_position",
        "trend_direction",
        "needs_refresh",
        "rule_prediction"
    ]
].head(10)

,content_id,ctr,engagement_rate,avg_position,trend_direction,needs_refresh,rule_prediction
1,content_a1fb4e703a9e,0.05,0.00,20.3,down,0,1
5,content_d4084a4bc775,0.03,0.00,8.5,down,0,1
6,content_9a34b442b552,0.00,0.00,7.0,down,0,1
7,content_a63219c6e95a,0.06,3.57,21.2,stable,0,1
11,content_5a3e876cf7f7,0.00,0.00,0.0,new,0,1
13,content_a5a2fbc76336,0.00,0.00,39.8,stable,0,1
14,content_91067a14431a,0.00,0.00,27.1,down,0,1
21,content_9d548144b06d,0.00,0.00,12.6,stable,0,1
25,content_033ae3e7aecf,0.00,0.00,7.2,down,0,1
28,content_19ad8f9bac29,0.00,0.00,59.3,down,0,1


# 6. Content Action Supported

The model output supports a real business decision.

Example Prediction:

Probability content needs refresh = 0.87

Possible Actions:

- Add content to a refresh queue
- Prioritize editorial review
- Schedule content updates
- Allocate resources to high-risk pages

The purpose of the model is not just prediction.

The purpose is to help content teams make better decisions about which content should be updated first.

## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.